# 🦾 J.A.R.V.I.S. HUD — Computer Vision Workshop

Welcome! By the end of this notebook you'll have a **live Iron Man-style HUD** running on your webcam.

### Rules of the road
- Run cells **in order** — Shift+Enter, or click the ▶ button
- Look for `# 🔧 YOUR TURN` — that's the only place you need to type anything
- If something breaks, raise your hand or re-run from the top

| Section | What happens | Time |
|---------|-------------|------|
| 1 | Install libraries | ~60 sec |
| 2 | See yourself on screen | ~1 min |
| 3 | Face detection + HUD | ~2 min |
| 4 | Make it yours | you control this |
| 5 | Bonus: health bar | if you finish early |

---
## 📦 Section 1 — Install & Import
Run this once. It takes about 60 seconds. You'll see `✅ Ready!` when it's done.

> ⚠️ If it asks you to **restart the runtime**, click yes — then run this cell again.

In [ ]:
# Pin mediapipe to a version that works with Colab
!pip install mediapipe==0.10.9 opencv-python-headless --quiet

import cv2
import numpy as np
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import time

# Import the new-style mediapipe face detector
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# Download the face detection model file from Google
import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
    "face_detector.tflite"
)

# Set up the detector
base_options = mp_python.BaseOptions(model_asset_path='face_detector.tflite')
options      = mp_vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)
detector     = mp_vision.FaceDetector.create_from_options(options)

# ── Webcam helper (no need to read this — just run it) ──────────────────────
def capture_frame():
    js = Javascript('''
        async function capture() {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await new Promise(r => video.onloadedmetadata = r);
            video.play();
            await new Promise(r => setTimeout(r, 500));
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(t => t.stop());
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        capture().then(result => element.textContent = result);
    ''')
    display(js)
    data_url  = eval_js('capture()')
    img_bytes = b64decode(data_url.split(',')[1])
    img_array = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def show_image(img, title=''):
    _, buffer = cv2.imencode('.jpg', img)
    display(IPImage(data=buffer.tobytes()))
    if title: print(f"👆 {title}")

print("✅ Ready!")

---
## 📸 Section 2 — Hello Webcam
Run this cell. Your browser will ask for camera permission — click **Allow**.

You should see a photo of yourself appear below the cell.

In [ ]:
print("📷 Smile! Capturing...")
frame = capture_frame()
show_image(frame, title=f"Your webcam feed — {frame.shape[1]}x{frame.shape[0]}px")

---
## 🟩 Section 2.5 — Draw Your First Rectangle

Before the AI does anything fancy, let's learn how OpenCV draws on images.
This is the same drawing tool the HUD uses — we're just practicing on a blank canvas first.

**`cv2.rectangle(image, top_left, bottom_right, color, thickness)`**

- `top_left` — (x, y) of the top-left corner
- `bottom_right` — (x, y) of the bottom-right corner
- `color` — (Blue, Green, Red) each 0–255
- `thickness` — border size in pixels. Use `-1` to fill it solid

> 🔧 **YOUR TURN:** Change the color and coordinates, then run it!

In [ ]:
# Create a blank black canvas — 400px tall, 600px wide
canvas = np.zeros((400, 600, 3), dtype=np.uint8)

# 🔧 YOUR TURN — change these!
TOP_LEFT     = (100, 100)   # 🔧 (x, y) of top-left corner
BOTTOM_RIGHT = (400, 300)   # 🔧 (x, y) of bottom-right corner
COLOR        = (0, 255, 0)  # 🔧 (Blue, Green, Red) — this is green
THICKNESS    = 3            # 🔧 try -1 to fill it solid!

# Draw the rectangle
cv2.rectangle(canvas, TOP_LEFT, BOTTOM_RIGHT, COLOR, THICKNESS)

show_image(canvas, title='My first rectangle!')

---
## 🤖 Section 3 — Face Detection + HUD

This cell does everything at once:
1. Captures a frame from your webcam
2. Runs a pre-trained AI model to find faces
3. Draws the Iron Man HUD on top

Just run it — **no changes needed yet**. Customization is in Section 4!

In [ ]:
# ── HUD Settings (you'll customize these in Section 4) ──────────────────────
HUD_COLOR    = (0, 255, 180)    # color of boxes & text  (B, G, R)
LABEL_BG     = (0, 80, 50)      # label background color (B, G, R)
STATUS_COLOR = (0, 200, 255)    # status bar text color  (B, G, R)
PILOT_NAME   = "PILOT: TONY STARK"
STATUS_MSG   = "JARVIS ACTIVE  |  THREAT LEVEL: LOW  |  SYSTEMS: NOMINAL"

# ── Drawing helpers ──────────────────────────────────────────────────────────
def draw_corners(img, x, y, w, h, color, length=22):
    pts = [
        ((x,   y),   (x+length, y),     (x,   y+length)),
        ((x+w, y),   (x+w-length, y),   (x+w, y+length)),
        ((x,   y+h), (x+length, y+h),   (x,   y+h-length)),
        ((x+w, y+h), (x+w-length, y+h), (x+w, y+h-length)),
    ]
    for corner, h_pt, v_pt in pts:
        cv2.line(img, corner, h_pt, color, 2)
        cv2.line(img, corner, v_pt, color, 2)

def draw_label(img, text, x, y, color, bg):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(text, font, 0.55, 1)
    cv2.rectangle(img, (x, y - th - 10), (x + tw + 10, y), bg, -1)
    cv2.putText(img, text, (x + 5, y - 5), font, 0.55, color, 1, cv2.LINE_AA)

def draw_status(img, msg, color):
    h, w = img.shape[:2]
    cv2.rectangle(img, (0, h - 30), (w, h), (0, 0, 0), -1)
    cv2.putText(img, msg, (10, h - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

def run_hud(pilot_name, hud_color, label_bg, status_color, status_msg):
    print("📷 Capturing frame...")
    frame  = capture_frame()
    h, w   = frame.shape[:2]
    output = frame.copy()

    # MediaPipe needs an RGB image wrapped in its own Image type
    rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image   = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    results    = detector.detect(mp_image)

    if results.detections:
        for det in results.detections:
            score = det.categories[0].score
            bbox  = det.bounding_box
            x, y  = bbox.origin_x, bbox.origin_y
            bw, bh = bbox.width, bbox.height

            # Clamp to frame bounds
            x  = max(0, x)
            y  = max(0, y)

            # Subtle tint inside the box
            overlay = output.copy()
            cv2.rectangle(overlay, (x, y), (x+bw, y+bh), hud_color, -1)
            cv2.addWeighted(overlay, 0.08, output, 0.92, 0, output)

            draw_corners(output, x, y, bw, bh, hud_color)
            draw_label(output, f"{pilot_name}  |  CONF: {score:.0%}", x, y, hud_color, label_bg)

        print(f"🎯 {len(results.detections)} face(s) detected — HUD online!")
    else:
        print("😅 No faces found — try better lighting or move closer.")

    draw_status(output, status_msg, status_color)
    show_image(output, title="J.A.R.V.I.S. HUD")

run_hud(PILOT_NAME, HUD_COLOR, LABEL_BG, STATUS_COLOR, STATUS_MSG)

---
## 🎨 Section 4 — Make It Yours

Change the values below, then run the cell. That's it!

**Colors** are written as `(Blue, Green, Red)` — each number is 0–255.

| Color | Value |
|-------|-------|
| Iron Man red | `(0, 0, 220)` |
| Arc reactor blue | `(255, 180, 0)` |
| War Machine silver | `(200, 200, 200)` |
| Rescue gold | `(0, 215, 255)` |
| Default cyan-green | `(0, 255, 180)` |

In [ ]:
# ════════════════════════════════════════════════
#   🔧 YOUR TURN — change anything below!
# ════════════════════════════════════════════════

HUD_COLOR    = (0, 255, 180)          # 🔧 try a different color!
LABEL_BG     = (0, 80, 50)            # 🔧 label background
STATUS_COLOR = (0, 200, 255)          # 🔧 status bar text

PILOT_NAME   = "PILOT: TONY STARK"   # 🔧 put YOUR name here!
STATUS_MSG   = "JARVIS ACTIVE  |  THREAT LEVEL: LOW  |  SYSTEMS: NOMINAL"
#                                      🔧 change the status message!

# ════════════════════════════════════════════════
run_hud(PILOT_NAME, HUD_COLOR, LABEL_BG, STATUS_COLOR, STATUS_MSG)

---
## ⭐ Section 5 — Bonus: Add a Health Bar

For fast finishers! Adds a power bar below each detected face.

Change `POWER_LEVEL` to any number between 0 and 100.

In [ ]:
# ════════════════════════════════════════════════
#   🔧 YOUR TURN
# ════════════════════════════════════════════════
POWER_LEVEL  = 87                     # 🔧 any number 0–100
PILOT_NAME   = "PILOT: TONY STARK"   # 🔧 your name
HUD_COLOR    = (0, 255, 180)          # 🔧 your color
LABEL_BG     = (0, 80, 50)
STATUS_COLOR = (0, 200, 255)
# ════════════════════════════════════════════════

def draw_power_bar(img, x, y, w, level, color):
    bar_y  = y + 6
    fill_w = int(w * (level / 100))
    cv2.rectangle(img, (x, bar_y), (x + w, bar_y + 8), (40, 40, 40), -1)
    cv2.rectangle(img, (x, bar_y), (x + fill_w, bar_y + 8), color, -1)
    cv2.putText(img, f"PWR {level}%", (x + w + 6, bar_y + 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

print("📷 Capturing...")
frame   = capture_frame()
h, w    = frame.shape[:2]
output  = frame.copy()

rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
results  = detector.detect(mp_image)

if results.detections:
    for det in results.detections:
        score  = det.categories[0].score
        bbox   = det.bounding_box
        x, y   = max(0, bbox.origin_x), max(0, bbox.origin_y)
        bw, bh = bbox.width, bbox.height

        overlay = output.copy()
        cv2.rectangle(overlay, (x, y), (x+bw, y+bh), HUD_COLOR, -1)
        cv2.addWeighted(overlay, 0.08, output, 0.92, 0, output)
        draw_corners(output, x, y, bw, bh, HUD_COLOR)
        draw_label(output, f"{PILOT_NAME}  |  CONF: {score:.0%}", x, y, HUD_COLOR, LABEL_BG)
        draw_power_bar(output, x, y + bh, bw, POWER_LEVEL, HUD_COLOR)

    print(f"🎯 {len(results.detections)} face(s) — suit at {POWER_LEVEL}% power!")
else:
    print("😅 No faces — try again!")

draw_status(output, f"POWER: {POWER_LEVEL}%  |  JARVIS ACTIVE  |  SYSTEMS: NOMINAL", STATUS_COLOR)
show_image(output, title=f"HUD + Power Bar ({POWER_LEVEL}%)")

---
## 🎉 You built it!

| What you did | How it works |
|---|---|
| Captured webcam frames | JavaScript → base64 → numpy array |
| Detected faces | MediaPipe model (trained on millions of images) |
| Drew the HUD | OpenCV rectangles, lines, and text |
| Customized it | Just changing Python variables |

### Want to go further?
- [MediaPipe](https://developers.google.com/mediapipe) — hand tracking, pose, iris detection
- [Ultralytics YOLOv8](https://docs.ultralytics.com) — detect any object (phones, cars, animals)
- [Roboflow](https://roboflow.com) — train a model on your OWN custom photos

---
*Computer Vision Workshop · Built with MediaPipe + OpenCV*